First creating an excel which lists all undetected instances.

In [ ]:
"""
undetected_bc_across_models.py
===============================
Lists custom_ids where the breaking change (BC) was NOT detected
(i.e., ALL generated tests passed on v2) across different models
and context variants.

Output: Long-format CSV with one row per (custom_id, model, context_variant).

Also produces a summary showing which custom_ids are undetected across
ALL models/variants vs. only some.
"""

import json
import csv
from pathlib import Path
from collections import defaultdict


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CONFIG — UPDATE THESE BEFORE EACH RUN
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Each entry: (model_name, context_variant, path_to_transplant_results_json)
# The JSON file should have the same structure as transplant_results_breaking_single_module.json
LLM_CONFIGS = [
    # --- GPT-4o ---
    ("GPT-4o", "Minimal", "/Volumes/Rachna-HD/GPTResults/Exp3BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("GPT-4o", "Method",  "/Volumes/Rachna-HD/GPTResults/Exp6BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("GPT-4o", "Class",   "/Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/bre/transplant_results_breaking_single_module.json"),

    # --- Qwen-480B ---
    ("Qwen-480B", "Minimal", "/Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("Qwen-480B", "Method",  "/Volumes/Rachna-HD/Qwen480Results/Exp6BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("Qwen-480B", "Class",   "/Volumes/Rachna-HD/Qwen480Results/Exp7BatchResults/bre/transplant_results_breaking_single_module.json"),

     # --- GPTOSS-120b ---
    ("GPTOSS-120B", "Minimal", "/Volumes/Rachna-HD/GPTOSSResults/Exp3BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("GPTOSS-120B", "Method",  "/Volumes/Rachna-HD/GPTOSSResults/Exp6BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("GPTOSS-120B", "Class",   "/Volumes/Rachna-HD/GPTOSSResults/Exp7BatchResults/bre/transplant_results_breaking_single_module.json"),
]

# BUMP CSV for ground-truth metadata
BUMP_CSV = "/Volumes/Rachna-HD/ConfigFiles/Candidate_BUMP_Instance_errorTypes.csv"

# Output files
OUTPUT_CSV       = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/MissedBC/undetected_bc_across_models.csv"
SUMMARY_CSV      = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/MissedBC/undetected_bc_summary.csv"

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# END CONFIG
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


def parse_bump_errors(raw: str) -> str:
    if not raw or str(raw).strip() in ('', 'nan'):
        return ''
    result = set()
    for e in str(raw).split('|'):
        e = e.strip()
        if not e or 'MojoFailureException' in e or 'MojoExecutionException' in e:
            continue
        result.add(e.split('.')[-1])
    return '|'.join(sorted(result))


def load_bump(bump_csv: str) -> dict:
    data = {}
    with open(bump_csv, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            cid = row['custom_id']
            data[cid] = {
                'bump_bc_errors':           parse_bump_errors(row.get('exception_types', '')),
                'bump_bc_errors_raw':       row.get('exception_types', ''),
                'failure_category':         row.get('failureCategory', ''),
            }
    return data


def load_llm_results(llm_json: str) -> dict:
    with open(llm_json) as f:
        data = json.load(f)
    return data.get('results', data)


def main():
    print("Loading BUMP data...")
    bump = load_bump(BUMP_CSV)
    print(f"  BUMP instances: {len(bump)}")

    rows = []
    # Track which (custom_id, model, variant) combos are undetected
    undetected_tracker = defaultdict(set)  # custom_id → set of (model, variant)
    total_configs = len(LLM_CONFIGS)

    for model_name, context_variant, json_path in LLM_CONFIGS:
        print(f"\nProcessing: {model_name} / {context_variant}")
        print(f"  File: {json_path}")

        if not Path(json_path).is_file():
            print(f"  WARNING: File not found — skipping")
            continue

        llm = load_llm_results(json_path)
        print(f"  Instances loaded: {len(llm)}")

        total_instances = 0
        detected_count = 0
        undetected_count = 0

        for instance_id, instance_data in llm.items():
            total_instances += 1
            tests_data   = instance_data.get('tests', {})
            passed_files = tests_data.get('passed', [])
            failed_tests = tests_data.get('failed', [])

            total_tests = len(passed_files) + len(failed_tests)
            num_detected = len(failed_tests)
            num_missed   = len(passed_files)

            # BC is undetected if there are NO failed tests (all passed on v2)
            bc_detected = num_detected > 0

            if bc_detected:
                detected_count += 1
            else:
                undetected_count += 1
                undetected_tracker[instance_id].add((model_name, context_variant))

                bump_info = bump.get(instance_id, {})

                rows.append({
                    'custom_id':                instance_id,
                    'model':                    model_name,
                    'context_variant':          context_variant,
                    'total_tests_generated':    total_tests,
                    'tests_passed_v2':          num_missed,
                    'tests_failed_v2':          num_detected,
                    'bc_detected':              False,
                    'bump_bc_failure_category': bump_info.get('failure_category', ''),
                    'bump_bc_errors':           bump_info.get('bump_bc_errors', ''),
                    'bump_bc_errors_raw':       bump_info.get('bump_bc_errors_raw', ''),
                })

        print(f"  BC detected:   {detected_count} / {total_instances}")
        print(f"  BC undetected: {undetected_count} / {total_instances}")

    # ── Write long-format output CSV ──────────────────────────────────────────
    Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        'custom_id', 'model', 'context_variant',
        'total_tests_generated', 'tests_passed_v2', 'tests_failed_v2', 'bc_detected',
        'bump_bc_failure_category', 'bump_bc_errors', 'bump_bc_errors_raw',
    ]
    with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    # ── Write summary CSV ─────────────────────────────────────────────────────
    # One row per custom_id: how many model/variant combos missed it
    Path(SUMMARY_CSV).parent.mkdir(parents=True, exist_ok=True)
    summary_rows = []
    for cid, configs in sorted(undetected_tracker.items()):
        bump_info = bump.get(cid, {})
        models_missed  = sorted(set(m for m, v in configs))
        variants_missed = sorted(set(v for m, v in configs))
        summary_rows.append({
            'custom_id':                    cid,
            'num_configs_undetected':       len(configs),
            'total_configs':                total_configs,
            'undetected_in_all_configs':    len(configs) == total_configs,
            'models_undetected':            '|'.join(models_missed),
            'variants_undetected':          '|'.join(variants_missed),
            'model_variant_pairs':          '|'.join(f"{m}+{v}" for m, v in sorted(configs)),
            'bump_bc_failure_category':     bump_info.get('failure_category', ''),
            'bump_bc_errors':               bump_info.get('bump_bc_errors', ''),
        })

    # Sort: instances undetected in ALL configs first, then by count descending
    summary_rows.sort(key=lambda r: (-r['num_configs_undetected'], r['custom_id']))

    summary_fields = [
        'custom_id', 'num_configs_undetected', 'total_configs', 'undetected_in_all_configs',
        'models_undetected', 'variants_undetected', 'model_variant_pairs',
        'bump_bc_failure_category', 'bump_bc_errors',
    ]
    with open(SUMMARY_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=summary_fields)
        writer.writeheader()
        writer.writerows(summary_rows)

    # ── Console summary ───────────────────────────────────────────────────────
    print(f"\n{'='*70}")
    print(f"UNDETECTED BC SUMMARY")
    print(f"{'='*70}")
    print(f"Total model/variant configs:        {total_configs}")
    print(f"Total undetected rows (long format): {len(rows)}")
    print(f"Unique custom_ids with ≥1 miss:     {len(undetected_tracker)}")

    all_missed = sum(1 for cid, cfgs in undetected_tracker.items() if len(cfgs) == total_configs)
    print(f"Undetected in ALL configs:          {all_missed}")
    print(f"Undetected in SOME configs:         {len(undetected_tracker) - all_missed}")

    print(f"\nPer-config undetected counts:")
    for model_name, context_variant, _ in LLM_CONFIGS:
        count = sum(1 for cfgs in undetected_tracker.values() if (model_name, context_variant) in cfgs)
        print(f"  {model_name:20s} {context_variant:10s}: {count}")

    print(f"\nOutput written to:")
    print(f"  Long-format CSV: {OUTPUT_CSV}")
    print(f"  Summary CSV:     {SUMMARY_CSV}")


if __name__ == "__main__":
    main()

Loading BUMP data...
  BUMP instances: 89

Processing: GPT-4o / Minimal
  File: /Volumes/Rachna-HD/GPTResults/Exp3BatchResults/bre/transplant_results_breaking_single_module.json
  Instances loaded: 38
  BC detected:   17 / 38
  BC undetected: 21 / 38

Processing: GPT-4o / Method
  File: /Volumes/Rachna-HD/GPTResults/Exp6BatchResults/bre/transplant_results_breaking_single_module.json
  Instances loaded: 32
  BC detected:   13 / 32
  BC undetected: 19 / 32

Processing: GPT-4o / Class
  File: /Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/bre/transplant_results_breaking_single_module.json
  Instances loaded: 31
  BC detected:   27 / 31
  BC undetected: 4 / 31

Processing: Qwen-480B / Minimal
  File: /Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/bre/transplant_results_breaking_single_module.json
  Instances loaded: 32
  BC detected:   18 / 32
  BC undetected: 14 / 32

Processing: Qwen-480B / Method
  File: /Volumes/Rachna-HD/Qwen480Results/Exp6BatchResults/bre/transplant_results_br

#  CONFIGURE PATHS — one folder per (model, context_variant)
# ═══════════════════════════════════════════════════════════════
FAILED_INSTANCES_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/MissedBC/undetected_bc_across_models.csv"
BUMP_LOGS_DIR        = "/Volumes/Rachna-HD/RQResults/RQ4/BumpExecutionlogs"          # <custom_id>.log

# Map every (model, context_variant) pair to its LLM log folder.
# Add / remove entries to match your actual folder layout.
LLM_LOG_DIRS: dict[tuple[str, str], str] = {
    ("GPT-4o",   "Minimal"): "/Volumes/Rachna-HD/GPTResults/Exp3BatchResults/bre/logs",
    ("GPT-4o",   "Method"):  "/Volumes/Rachna-HD/GPTResults/Exp6BatchResults/bre/logs",
    ("GPT-4o",   "Class"):   "/Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/bre/logs",
    ("Qwen-480B", "Minimal"): "/Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/bre/",
    ("Qwen-480B", "Method"):  "/Volumes/Rachna-HD/Qwen480Results/Exp6BatchResults/bre/",
    ("Qwen-480B", "Class"):   "/Volumes/Rachna-HD/Qwen480Results/Exp7BatchResults/bre/",
    ("GPTOSS-120B",   "Minimal"): "/Volumes/Rachna-HD/GPTOSSResults/Exp3BatchResults/bre/",
    ("GPTOSS-120B",   "Method"):  "/Volumes/Rachna-HD/GPTOSSResults/Exp6BatchResults/bre/",
    ("GPTOSS-120B",   "Class"):   "/Volumes/Rachna-HD/GPTOSSResults/Exp7BatchResults/bre/",
}

OUTPUT_CSV  = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/MissedBC/missedAPIcoverage_analysis.csv"
# ═══════════════════════════════════════════════════════════════

In [11]:
"""
BUMP Breaking API Coverage Analyzer — Single CSV Output
=========================================================================
"""

import csv
import logging
import re
from pathlib import Path
from collections import defaultdict

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
log = logging.getLogger(__name__)

# ═══════════════════════════════════════════════════════════════
#  CONFIGURE PATHS  — edit these before running
# ═══════════════════════════════════════════════════════════════
INPUT_CSV   = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/MissedBC/ManualBrokenAPICodingBumpUndetected.csv"
OUTPUT_DIR  = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/MissedBC/"

LLM_LOG_DIRS: dict[tuple[str, str], str] = {
    ("GPT-4o",      "Minimal"): "/Volumes/Rachna-HD/GPTResults/Exp3BatchResults/bre/logs",
    ("GPT-4o",      "Method"):  "/Volumes/Rachna-HD/GPTResults/Exp6BatchResults/bre/logs",
    ("GPT-4o",      "Class"):   "/Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/bre/logs",
    ("Qwen-480B",   "Minimal"): "/Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/bre/logs",
    ("Qwen-480B",   "Method"):  "/Volumes/Rachna-HD/Qwen480Results/Exp6BatchResults/bre/logs",
    ("Qwen-480B",   "Class"):   "/Volumes/Rachna-HD/Qwen480Results/Exp7BatchResults/bre/logs",
    ("GPTOSS-120B", "Minimal"): "/Volumes/Rachna-HD/GPTOSSResults/Exp3BatchResults/bre/logs",
    ("GPTOSS-120B", "Method"):  "/Volumes/Rachna-HD/GPTOSSResults/Exp6BatchResults/bre/logs",
    ("GPTOSS-120B", "Class"):   "/Volumes/Rachna-HD/GPTOSSResults/Exp7BatchResults/bre/logs",
}
# ═══════════════════════════════════════════════════════════════

# ── helpers ────────────────────────────────────────────────────────────────────

def parse_tokens(pipe_str: str) -> list[str]:
    """Split pipe-separated tokens, strip whitespace, drop empty strings."""
    return [t.strip() for t in (pipe_str or "").split("|") if t.strip()]

def find_llm_logs(folder: str, custom_id: str) -> list[Path]:
    """Glob for all LLM run logs for this custom_id."""
    return sorted(Path(folder).glob(f"{custom_id}_*_breaking_single.log"))

def tokens_found_in_logs(log_paths: list[Path], tokens: list[str]) -> tuple[bool, list[str]]:
    hits: set[str] = set()
    
    for p in log_paths:
        try:
            text = p.read_text(encoding="utf-8", errors="replace")
            # Pre-compute lowercase text for case-insensitive fallback checks
            text_lower = text.lower()
            
            for t in tokens:
                # 1. Reliable Exact Match (Case-sensitive & Case-insensitive)
                if t in text or t.lower() in text_lower:
                    hits.add(t)
                    continue
                    
                # 2. Extract simple name (e.g., org.foo.Bar$Inner -> Bar$Inner)
                simple = t.rsplit(".", 1)[-1]
                
                # 3. Handle Inner Classes (Bytecode '$' vs Source '.')
                if "$" in simple:
                    source_level = simple.replace("$", ".")
                    if source_level in text or source_level.lower() in text_lower:
                        hits.add(t)
                        continue
                    # Check just the innermost class name
                    inner_only = simple.rsplit("$", 1)[-1]
                    if inner_only and re.search(r'\b' + re.escape(inner_only) + r'\b', text, re.IGNORECASE):
                        hits.add(t)
                        continue

                # 4. Handle Methods (Strip parameters to allow LLM parameter additions)
                if "(" in simple:
                    base_name = simple.split("(")[0].strip()
                    if not base_name: 
                        continue
                        
                    # Use \b for normal methods, (?<!\w) for special ones like <init>
                    if re.match(r'^\w+$', base_name):
                        pattern = r'\b' + re.escape(base_name) + r'\s*\('
                    else:
                        pattern = r'(?<!\w)' + re.escape(base_name) + r'\s*\('
                        
                    if re.search(pattern, text, re.IGNORECASE):
                        hits.add(t)
                    continue
                    
                # 5. Standard Class/Field Fallback
                if simple in text or simple.lower() in text_lower:
                    if re.match(r'^\w+$', simple):
                        # Enforce boundaries so "End" doesn't match "Backend"
                        if re.search(r'\b' + re.escape(simple) + r'\b', text, re.IGNORECASE):
                            hits.add(t)
                    else:
                        # Catch generics like List<String> or arrays String[]
                        hits.add(t)
                        
        except Exception as e:
            log.warning(f"Could not read log file {p}: {e}")
            
    return bool(hits), sorted(hits)

def evaluate_coverage(log_paths: list[Path], tokens: list[str], folder: str, custom_id: str, token_type: str) -> tuple[str, str, str]:
    """Returns (exercised, matched_tokens_str, miss_reason)"""
    if not tokens:
        return "No_ground_truth", "", f"No ground-truth tokens coded in CSV for {token_type}"
    if not log_paths:
        if not folder:
            return "No_logs", "", "Model/Variant pair not found in LLM_LOG_DIRS"
        return "No_logs", "", f"No LLM log files found in {folder} for {custom_id}"
    
    covered, tokens_hit = tokens_found_in_logs(log_paths, tokens)
    if covered:
        return "Yes", "|".join(tokens_hit), "LLM tests DID exercise the token"
    return "No", "", "LLM tests did NOT exercise the broken token at all"

# ── unified analysis ───────────────────────────────────────────────────────────

def analyze_combined(undetected_rows: list[dict], llm_dirs: dict) -> list[dict]:
    """Run coverage analysis for both client class and OSS API simultaneously."""
    results = []
    
    normalized_dirs = {
        (str(k[0]).strip().lower(), str(k[1]).strip().lower()): v 
        for k, v in llm_dirs.items()
    }

    for row in undetected_rows:
        out_row = row.copy()

        custom_id   = row.get("custom_id", "").strip()
        model       = row.get("model", "").strip()
        ctx_variant = row.get("context_variant", "").strip()

        folder = normalized_dirs.get((model.lower(), ctx_variant.lower()))
        log_paths = find_llm_logs(folder, custom_id) if folder else []

        # Evaluate Client Class
        client_tokens = parse_tokens(row.get("Client_class_in_test", ""))
        c_exercised, c_matched, c_reason = evaluate_coverage(log_paths, client_tokens, folder, custom_id, "Client_class_in_test")

        # Evaluate OSS API
        oss_tokens = parse_tokens(row.get("Broken_oss_API", ""))
        o_exercised, o_matched, o_reason = evaluate_coverage(log_paths, oss_tokens, folder, custom_id, "Broken_oss_API")

        # Append tracking fields
        out_row["llm_logs_found"]        = len(log_paths)
        out_row["client_exercised"]      = c_exercised
        out_row["client_tokens_matched"] = c_matched
        out_row["client_miss_reason"]    = c_reason
        out_row["oss_exercised"]         = o_exercised
        out_row["oss_tokens_matched"]    = o_matched
        out_row["oss_miss_reason"]       = o_reason

        results.append(out_row)

    return results

# ── CSV writer ─────────────────────────────────────────────────────────────────

def write_csv(rows: list[dict], path: str, original_fieldnames: list[str]) -> None:
    if not rows:
        log.warning(f"No data to write for {path}!")
        return

    new_fields = [
        "llm_logs_found", 
        "client_exercised", "client_tokens_matched", "client_miss_reason",
        "oss_exercised", "oss_tokens_matched", "oss_miss_reason"
    ]
    
    all_fields = list(dict.fromkeys(original_fieldnames + new_fields))

    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=all_fields, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)
            
    log.info(f"Written → {path} ({len(rows)} rows)")

# ── summary printer ────────────────────────────────────────────────────────────

def print_summary(label: str, rows: list[dict], target_key: str) -> None:
    print(f"\n{'='*70}")
    print(f"  {label}")
    print(f"{'='*70}")

    if not rows:
        print("  No data processed.")
        return

    total     = len(rows)
    yes       = sum(1 for r in rows if r.get(target_key) == "Yes")
    no        = sum(1 for r in rows if r.get(target_key) == "No")
    no_logs   = sum(1 for r in rows if r.get(target_key) == "No_logs")
    no_gt     = sum(1 for r in rows if r.get(target_key) == "No_ground_truth")
    
    print(f"  Total undetected rows : {total}")
    print(f"  No LLM logs           : {no_logs}")
    print(f"  No ground truth       : {no_gt}")

    by_pair: dict[tuple, list[dict]] = defaultdict(list)
    for r in rows:
        by_pair[(r.get("model", ""), r.get("context_variant", ""))].append(r)

    print(f"\n  {'Model':<14} {'Variant':<10} {'N':>5}  "
          f"{'Yes':>6}  {'No':>6}  {'NoLogs':>7}  {'NoGT':>5}")
    print("  " + "-"*56)
    for (model, variant), grp in sorted(by_pair.items()):
        n   = len(grp)
        y   = sum(1 for r in grp if r.get(target_key) == "Yes")
        nn  = sum(1 for r in grp if r.get(target_key) == "No")
        nl  = sum(1 for r in grp if r.get(target_key) == "No_logs")
        ng  = sum(1 for r in grp if r.get(target_key) == "No_ground_truth")
        print(f"  {model:<14} {variant:<10} {n:>5}  {y:>6}  {nn:>6}  {nl:>7}  {ng:>5}")
    print("")

# ── main ───────────────────────────────────────────────────────────────────────

def analyze(
    input_csv:  str  = INPUT_CSV,
    output_dir: str  = OUTPUT_DIR,
    llm_dirs:   dict = None,
) -> None:
    if llm_dirs is None:
        llm_dirs = LLM_LOG_DIRS

    try:
        with open(input_csv, newline="", encoding="utf-8-sig") as f:
            all_rows = list(csv.DictReader(f))
    except FileNotFoundError:
        log.error(f"Could not find input CSV at: {input_csv}")
        return

    if not all_rows:
        log.error("Input CSV is completely empty!")
        return

    original_columns = list(all_rows[0].keys())
    
    if "bc_detected" not in original_columns:
        log.warning(f"Column 'bc_detected' not found! Processing ALL rows.")
        undetected = all_rows
    else:
        undetected = [r for r in all_rows if r.get("bc_detected", "").strip().upper() == "FALSE"]

    log.info(f"Undetected rows: {len(undetected)} / {len(all_rows)}")

    if not undetected:
        log.error("No undetected rows found. Halting execution.")
        return

    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    # 1. Generate Combined Data
    combined_results = analyze_combined(undetected, llm_dirs)
    
    # 2. Write the SINGLE Master CSV
    write_csv(combined_results, str(out / "combined_coverage.csv"), original_columns)
    
    # 3. Print Summaries
    print_summary("CLIENT CLASS: Was it exercised by LLM tests?", combined_results, "client_exercised")
    print_summary("OSS API: Was it exercised by LLM tests?", combined_results, "oss_exercised")

if __name__ == "__main__":
    analyze()

INFO: Undetected rows: 111 / 111
INFO: Written → /Volumes/Rachna-HD/RQResultsForPaper/RQ3/MissedBC/combined_coverage.csv (111 rows)



  CLIENT CLASS: Was it exercised by LLM tests?
  Total undetected rows : 111
  No LLM logs           : 0
  No ground truth       : 1

  Model          Variant        N     Yes      No   NoLogs   NoGT
  --------------------------------------------------------
  GPT-4o         Class          4       1       3        0      0
  GPT-4o         Method        19       1      18        0      0
  GPT-4o         Minimal       21       0      21        0      0
  GPTOSS-120B    Class         11       0      11        0      0
  GPTOSS-120B    Method         9       0       9        0      0
  GPTOSS-120B    Minimal        5       0       5        0      0
  Qwen-480B      Class         14       1      12        0      1
  Qwen-480B      Method        14       0      14        0      0
  Qwen-480B      Minimal       14       0      14        0      0


  OSS API: Was it exercised by LLM tests?
  Total undetected rows : 111
  No LLM logs           : 0
  No ground truth       : 0

  Model        